In [ ]:
# Toda célula de um caderno marimo é uma função Python; o que ela devolve no "return" fica para as outras células.
import marimo as mo

In [ ]:
# O PAINEL: a história em português simples e, numa caixa fechada, os detalhes técnicos (lê o PIPELINE.md).
import sys as _sys
_pasta = str(mo.notebook_dir())
if _pasta not in _sys.path:
    _sys.path.insert(0, _pasta)
import painel
painel.mostrar(mo)

# Ciclo 2, etapa 4: limpar

**Em português:** limpar não é apagar o que atrapalha; é tirar da **medida** o que não é independente e marcar o que
é duvidoso, para medir com e sem. Neste ciclo, a limpeza tinha uma tarefa a mais, decidida no contrato: refazer os
rótulos "mesmo fato" de fonte com mais de um mês, agora com a data à vista. Foram duas tentativas, e as duas pararam
pela regra escrita antes. Os rótulos ficam os originais, e esses pares viram uma fatia para medir também sem eles.

| O que fazemos | Em português | Em termos técnicos |
|---|---|---|
| **As regras do ciclo 1** | os pares que os juízes viram com a resposta e os 22 inventados ensinam, mas não medem | R1 e R2 de `prepare.pares_do_decisor` |
| **O re-julgamento** | duas tentativas de refazer os rótulos de fonte velha com a data; as duas pararam na sua conferência | `src/rejulgar_com_data.py`, `outputs/45` e `46` |
| **A fatia nova** | os pares "mesmo" de fonte com mais de um mês ficam marcados, para medir sem eles | a medir nas etapas 12 a 14 |

Nada aqui ajusta modelo: é contagem e marcação.

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SRC = mo.notebook_dir().parent / "challenges" / "challenge-01-desinformacao-v2" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import prepare
import guard

# A pilha de estudo do ciclo 2 (rodadas 1 a 3), com a limpeza das regras R1 a R5 e a idade da fonte.
estudo = prepare.tabela_do_decisor((1, 2, 3))
# A marca da fatia nova: "mesmo" com a fonte de mais de um mês (os rótulos que o re-julgamento não conseguiu refazer).
estudo["mesmo_de_fonte_velha"] = (estudo["mesmo"] == True) & (estudo["idade_dias"] > 30)   # noqa: E712
INTERIM = SRC.parent / "data" / "interim"

In [ ]:
# FIGURA 1: QUEM ENTRA NA MEDIDA. Por rodada: pares que medem, pares que só ensinam, e a fatia de fonte velha.
_tabela = pd.DataFrame({
    "medem (a fatia de fonte velha à parte)": (estudo["usar_na_medida"] & ~estudo["mesmo_de_fonte_velha"]).groupby(estudo["rodada"]).sum(),
    '"mesmo" de fonte velha (medem; fatia de apoio)': (estudo["usar_na_medida"] & estudo["mesmo_de_fonte_velha"]).groupby(estudo["rodada"]).sum(),
    "só ensinam (vistos com a resposta, inventados)": (~estudo["usar_na_medida"]).groupby(estudo["rodada"]).sum(),
})
_fig, _ax = plt.subplots(figsize=(8, 3.6))
_tabela.plot.barh(stacked=True, ax=_ax)
_ax.set_xlabel("pares")
_ax.set_ylabel("rodada")
_ax.set_title("Quem entra na medida, depois da limpeza (estudo do ciclo 2)")
_ax.legend(fontsize=7, loc="lower right")
_milhar = lambda n: f"{n:,}".replace(",", ".")          # 10294 -> "10.294"
_total = int(estudo["usar_na_medida"].sum())
_velhos = int(estudo["mesmo_de_fonte_velha"].sum())
_so = int((~estudo["usar_na_medida"]).sum())
_explicacao = (
    f"Depois da limpeza, {_milhar(_total)} dos {_milhar(len(estudo))} pares de estudo entram na medida e {_so} só ensinam (os vistos "
    f"pelos juízes com a resposta e os inventados). Dentro dos que medem, {_velhos} são \"mesmo\" de fonte com mais de "
    "um mês: rótulos dados sem a data, que o re-julgamento não conseguiu refazer; toda medida sai também sem eles."
)
figura_medida = guard.fig(4, "quem_entra_na_medida", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_medida), width=620), mo.md(_explicacao), _tabela])

In [ ]:
# FIGURA 2: O PLACAR DAS DUAS TENTATIVAS DE RE-JULGAMENTO, só no estudo (a prova não se olha).
import conferir_rotulos as cr
MESMO = {"mesmo_acontecimento", "detalhe_divergente"}
_linhas = []
for _n, _pasta in [(1, INTERIM / "rejulgamento-data"), (2, INTERIM / "rejulgamento-data-2")]:
    _novo = pd.read_json(_pasta / "rotulo-novo.jsonl", lines=True).set_index("id")
    _oculto = pd.read_json(_pasta / "gabarito-oculto.jsonl", lines=True).set_index("id")
    _novo = _novo[_oculto.reindex(_novo.index)["lado_ciclo_2"] == "estudo"]
    _ancora = cr.carregar_tabela_md(_pasta / "ancora.md", "relação")
    _discorda = sum((letra in MESMO) != bool(_novo.loc[pid, "mesmo"]) for pid, letra in _ancora.items())
    _linhas.append({
        "tentativa": f"tentativa {_n}",
        "passam para \"não é o mesmo fato\"": 100 * ((_novo["origem"] == "re-julgado com a data") & ~_novo["mesmo"]).mean(),
        "os dois juízes concordam": 100 * (_novo["origem"] == "re-julgado com a data").mean(),
        "você concorda com o rótulo novo": 10 * (len(_ancora) - _discorda),
    })
placar = pd.DataFrame(_linhas).set_index("tentativa").round(1)
_fig, _ax = plt.subplots(figsize=(8, 3.6))
placar.T.plot.bar(ax=_ax, rot=0)
_ax.axhline(70, color="black", linestyle="--", linewidth=1)
_ax.text(2.45, 71, "mínimo da sua\nconferência (7 de 10)", fontsize=7, ha="right")
_ax.set_ylabel("de cada 100")
_ax.set_title("Re-julgar os rótulos de fonte velha com a data: duas tentativas (estudo)")
_ax.tick_params(axis="x", labelsize=7)
_p = placar
_explicacao = (
    f"Na tentativa 1 (só a data à vista), os juízes passaram {_p.iloc[0, 0]:.0f} de cada 100 para \"não é o mesmo fato\" e "
    f"você concordou com o rótulo novo em {_p.iloc[0, 2]:.0f} de 100; na tentativa 2 (a regra \"fato que se repete x fato que "
    f"acontece uma vez\"), {_p.iloc[1, 0]:.0f} passaram, os juízes concordaram entre si em {_p.iloc[1, 1]:.0f} e você em "
    f"{_p.iloc[1, 2]:.0f}. As duas abaixo do mínimo de 70: com fonte velha, \"é o mesmo fato?\" não tem resposta firme, nem "
    "para os juízes nem para o humano; os rótulos ficam os originais."
)
figura_rejulgamento = guard.fig(4, "rejulgamento_com_data", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_rejulgamento), width=620), mo.md(_explicacao), placar])

# Ciclo 2, etapa 5: juntar as informações

**Em português:** cada par precisa de uma ficha com tudo o que o app sabe na hora: a semelhança, a data da fonte, de
que buscador veio, o que a regra concluiu. Essas informações moram em arquivos diferentes, e juntar arquivos é onde
mais se perde ou duplica linha sem perceber. As junções são as do ciclo 1; o que muda é que a rodada 3 inteira entra no
estudo e a rodada 4 (a prova) também precisa da ficha completa.

| O que conferimos | Em português | Em termos técnicos |
|---|---|---|
| **Nenhuma linha perdida** | cada par entra uma vez, e nenhum some na junção | contagem antes e depois, ids únicos |
| **Quanto de cada informação existe** | por rodada: com data, com buscador, com comparação da regra, com trecho | cobertura das entradas |
| **A prova tem ficha completa** | a rodada 4 tem todas as entradas, inclusive a semelhança | entradas da rodada 4 (só presença; nada de rótulo) |

**O que eu espero, escrito antes de rodar:** zero linha perdida ou duplicada; na rodada 3, data em uns 74 de cada 100 e
comparação da regra em uns 24 (números do ciclo 1); na rodada 4, capturada com a mesma configuração, data entre 60 e
80 de cada 100, comparação da regra entre 15 e 35, e semelhança em todos.

In [ ]:
import prepare as _prepare

# 1. NENHUMA LINHA PERDIDA: a tabela juntada tem exatamente os pares limpos, uma vez cada.
_limpos = _prepare.pares_do_decisor((1, 2, 3))
_prova = _prepare.entradas_do_decisor((4,))                     # a ficha da prova: só presença, nenhum rótulo olhado
juncao = {
    "pares limpos (estudo)": len(_limpos),
    "pares na tabela juntada (estudo)": len(estudo),
    "ids repetidos na tabela juntada": int(estudo.index.duplicated().sum()),
    "pares da prova com ficha": len(_prova),
    "pares da prova sem semelhança": int(_prova["cosseno"].isna().sum()),
}
assert juncao["pares limpos (estudo)"] == juncao["pares na tabela juntada (estudo)"], "a junção mudou o número de pares"
assert juncao["ids repetidos na tabela juntada"] == 0

# 2. QUANTO DE CADA INFORMAÇÃO EXISTE, de cada 100 pares, por rodada (a 4 entra só pela presença).
def _cobertura(t):
    return pd.Series({
        "com data da fonte": 100 * t["idade_dias"].notna().mean(),
        "com buscador": 100 * (t["buscador"] != "ausente").mean(),
        "a regra comparou com cuidado": 100 * t["comparou"].mean(),
        "com trecho da página": 100 * t["trecho"].notna().mean(),
    })
_tab_prova = _prepare.tabela_do_decisor((4,)).assign(buscador=lambda d: d["buscador"].fillna("ausente"))
cobertura = pd.concat({**{f"rodada {r}": _cobertura(g.assign(buscador=g["buscador"].fillna("ausente")))
                          for r, g in estudo.groupby("rodada")},
                       "rodada 4 (prova)": _cobertura(_tab_prova)}, axis=1).round(1)

_fig, _ax = plt.subplots(figsize=(8, 3.8))
cobertura.T.plot.bar(ax=_ax, rot=0)
_ax.set_ylabel("de cada 100 pares")
_ax.set_title("Quanto de cada informação existe, por rodada")
_ax.legend(fontsize=7)
_c = cobertura
_explicacao = (
    f"A data da fonte existe em {_c.loc['com data da fonte', 'rodada 3']:.0f} de cada 100 pares da rodada 3 e em "
    f"{_c.loc['com data da fonte', 'rodada 4 (prova)']:.0f} da rodada 4; a regra comparou com cuidado em "
    f"{_c.loc['a regra comparou com cuidado', 'rodada 3']:.0f} e {_c.loc['a regra comparou com cuidado', 'rodada 4 (prova)']:.0f}. "
    "A rodada 1 não tem dump (sem data do buscador, sem buscador, sem trecho): é a mesma diferença de população da etapa 1. "
    f"Nenhuma linha perdida na junção ({juncao['pares na tabela juntada (estudo)']} pares de estudo, ids únicos)."
)
figura_cobertura = guard.fig(5, "cobertura_das_entradas", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_cobertura), width=620), mo.md(_explicacao), juncao, cobertura])

# Ciclo 2, etapa 6: separar estudo, simulado e prova

**Em português:** três montes. O **estudo** é o caderno de exercícios, onde o computador aprende. O **simulado** é onde
escolhemos entre modelos e a nota de corte. A **prova** fica no cofre e é aberta uma vez, no fim. Regra de ouro:
fontes da mesma notícia ficam sempre no mesmo monte, senão o computador decora em vez de aprender.

| Monte | O que é | Por quê |
|---|---|---|
| **Estudo** | rodadas 1 e 2 e 75% das notícias da rodada 3 (a antiga prova do ciclo 1 incluída) | mais material de onde aprender |
| **Simulado** | 25% das notícias da rodada 3, sorteadas por notícia inteira | a mesma época e a mesma configuração do app da prova; é onde o corte é escolhido |
| **Prova** | a rodada 4 inteira (lacrada antes das buscas), congelada no cofre do pipeline | só relata, uma vez, na etapa 14 |

**O cofre:** como no ciclo 1, congelamos uma tabela mínima da prova (o id do par e o rótulo) com a impressão digital
do pipeline; o `guard.final_test` só abre se a tabela for idêntica, e uma vez. A figura mostra o tamanho da prova, nunca
a proporção de "mesmo" dela.

**O que eu espero, escrito antes de rodar:** simulado com uns 50 acontecimentos e de 2.000 a 2.500 pares, com "mesmo"
entre 9 e 13 de cada 100 (a rodada 3 tem 11); estudo com 7.800 a 8.300 pares; prova com 4.298 pares e 100 notícias;
nenhum acontecimento em dois montes.

In [ ]:
import prepare as _prep

SEMENTE = 20261008                 # a mesma do ciclo 1: o sorteio se repete igual em qualquer máquina
FRACAO_SIMULADO = 0.25             # a mesma do ciclo 1

# Os acontecimentos (grupos) SÓ da rodada 3; um grupo que também tem pares de outra rodada (a regra R5) vai inteiro
# para o estudo, para o simulado ser só rodada 3. sorted() fixa a ordem antes do sorteio.
_rodadas_do_grupo = estudo.groupby("grupo")["rodada"].agg(set)
_grupos_r3 = sorted(g for g, r in _rodadas_do_grupo.items() if r == {3})
_rng = np.random.default_rng(SEMENTE)
_embaralhados = list(_rng.permutation(_grupos_r3))
grupos_simulado = sorted(_embaralhados[:int(round(len(_embaralhados) * FRACAO_SIMULADO))])
_no_simulado = estudo["grupo"].isin(grupos_simulado)
ids_simulado = sorted(estudo.index[_no_simulado])
ids_estudo = sorted(estudo.index[~_no_simulado])

# A PROVA: a rodada 4 inteira. A tabela mínima congelada: id e rótulo, ordenada pelo id.
_r4 = _prep.pares_do_decisor((4,))
prova = _r4[["mesmo"]].reset_index().sort_values("id", kind="stable").reset_index(drop=True)

# CHECAGENS: nenhum par em dois montes (a função do pipeline) e nenhum acontecimento em dois montes.
guard.check_no_overlap(pd.DataFrame({"id": ids_estudo}), pd.DataFrame({"id": ids_simulado}), prova[["id"]])
_g_e, _g_s, _g_p = set(estudo.loc[ids_estudo, "grupo"]), set(estudo.loc[ids_simulado, "grupo"]), set(_r4["grupo"])
assert not (_g_e & _g_s) and not (_g_e & _g_p) and not (_g_s & _g_p), "acontecimento em dois montes"
assert len(ids_estudo) + len(ids_simulado) == len(estudo)

# O COFRE do ciclo 2: o formato do guard.split, com a impressão digital da rodada 4. Nunca regravado.
_estado = mo.notebook_dir() / guard.STATE_FILE
_novo = {"test_fingerprint": guard._fingerprint(prova), "n_test": int(len(prova)), "test_touches": 0,
         "duplicates_dropped": 0, "created": "2026-10-08",
         "split": "rodada 4 inteira (lacre.json, commit 02d2798, antes das buscas), prova do ciclo 2; contrato aprovado no portão A"}
if _estado.is_file():
    assert json.loads(_estado.read_text(encoding="utf-8"))["test_fingerprint"] == _novo["test_fingerprint"], "o lacre mudou"
else:
    _estado.write_text(json.dumps(_novo, indent=2, ensure_ascii=False), encoding="utf-8")
_divisao = {"semente": SEMENTE, "fracao_simulado": FRACAO_SIMULADO, "grupos_simulado": grupos_simulado,
            "estudo": ids_estudo, "simulado": ids_simulado, "prova": prova["id"].tolist()}
_arq = mo.notebook_dir() / "divisao.json"
if _arq.is_file():
    assert json.loads(_arq.read_text(encoding="utf-8")) == _divisao, "a divisão mudou desde que foi gravada"
else:
    _arq.write_text(json.dumps(_divisao, indent=1, ensure_ascii=False), encoding="utf-8")

# O RETRATO: pares, acontecimentos e "mesmo" no estudo e no simulado; da prova, só o tamanho.
_linhas = []
for _nome, _ids in [("estudo", ids_estudo), ("simulado", ids_simulado)]:
    _t = estudo.loc[_ids]
    _linhas.append({"monte": _nome, "pares": len(_t), "acontecimentos": _t["grupo"].nunique(),
                    "mesmo de cada 100": round(100 * float(_t["mesmo"].mean()), 1)})
_linhas.append({"monte": "prova (cofre)", "pares": len(prova), "acontecimentos": len(_g_p), "mesmo de cada 100": None})
retrato = pd.DataFrame(_linhas).set_index("monte")
_fig, _ax = plt.subplots(figsize=(7, 2.8))
_ax.barh(retrato.index, retrato["pares"])
_ax.invert_yaxis()
_ax.set_xlabel("pares")
_ax.set_title("Estudo, simulado e prova (a proporção de \"mesmo\" da prova não é olhada)")
_e, _s = retrato.loc["estudo"], retrato.loc["simulado"]
_explicacao = (
    f"Estudo com {int(_e['pares'])} pares ({_e['mesmo de cada 100']:.1f} \"mesmo\" de cada 100), simulado com "
    f"{int(_s['pares'])} pares de {int(_s['acontecimentos'])} acontecimentos da rodada 3 ({_s['mesmo de cada 100']:.1f} de cada 100) "
    f"e prova com {len(prova)} pares de {len(_g_p)} notícias da rodada 4, no cofre. Nenhum acontecimento em dois montes. "
    "O simulado tem a época e a configuração do app da prova: é nele que o corte vai ser escolhido."
)
figura_divisao = guard.fig(6, "estudo_simulado_prova", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_divisao), width=560), mo.md(_explicacao), retrato])

# Ciclo 2, etapa 7: o que o computador enxerga

**Em português:** cada fonte vira uma ficha, e o computador só pode olhar o que o app sabe na hora de decidir. As 12
informações do ciclo 1 continuam. Entra uma candidata nova, que junta duas que o modelo já via separadas: **quão
parecida** a fonte é com a notícia e **quão velha** ela é. Separadas, as duas não contam ao modelo que "parecida e
velha" é justamente onde mora o engano (a figura abaixo mostra isso). A resposta final da regra segue candidata.

| Informação | O que é | Tipo |
|---|---|---|
| `cosseno` | semelhança de sentido afirmação × título, pelo codificador do app | número |
| `sobreposicao` | fração de palavras em comum (3 letras ou mais) | número de 0 a 1 |
| `idade_dias`, `tem_idade` | dias entre a fonte e a matéria, e se há data | número + marca |
| `comparou`, `titulo_cortado` | a regra comparou com cuidado; o título veio cortado | 0 ou 1 |
| `quem`, `acao`, `assunto`, `quando` | as partes da regra ("sem" quando não comparou) | categoria |
| `buscador`, `tipo_afirmacao` | de onde veio a fonte; o tipo que o app deu à afirmação | categoria |
| **`cosseno × idade` (nova, candidata)** | a semelhança multiplicada pela idade, as duas já padronizadas | número, **calculado na etapa 8** (usa a mediana e a escala aprendidas no estudo) |
| `regra` (candidata) | a resposta final da regra | categoria |

**Nunca entram:** a rodada, quem rotulou, a origem do rótulo, o veículo (o modelo decoraria sites) e tudo o que o app
não sabe na hora (lista `PROIBIDAS` do `prepare.py`).

**O que eu espero, escrito antes de rodar (só no estudo):** entre as fontes muito parecidas (cosseno de 0,74 ou mais),
são o mesmo fato mais de 90 de cada 100 no mesmo dia ou véspera, de 55 a 85 de 2 a 30 dias e menos de 50 acima de um
mês; entre as pouco parecidas (abaixo de 0,5), menos de 5 em qualquer idade. Se as faixas de idade não mudarem nada
entre as muito parecidas, a informação nova não tem o que ensinar.

In [ ]:
import prepare as _p2

# As entradas das rodadas 1 a 3, nos montes da etapa 6 (os mesmos ids, em qualquer máquina).
_tudo = _p2.entradas_do_decisor((1, 2, 3))
_div = json.loads((mo.notebook_dir() / "divisao.json").read_text(encoding="utf-8"))
estudo_entradas = _tudo.loc[_div["estudo"]]
simulado_entradas = _tudo.loc[_div["simulado"]]
ENTRADAS = _p2.ENTRADAS_NUMERICAS + _p2.ENTRADAS_CATEGORICAS + _p2.ENTRADA_CANDIDATA
assert not set(ENTRADAS) & set(_p2.PROIBIDAS), "entrada proibida"

# A FIGURA, SÓ COM O ESTUDO: de cada 100 fontes, quantas são o mesmo fato, por faixa de semelhança e de idade.
_e = estudo_entradas
_faixa_cos = pd.cut(_e["cosseno"], [-1, 0.5, 0.74, 1.01], labels=["pouco parecida (< 0,50)", "média (0,50 a 0,74)", "muito parecida (≥ 0,74)"], right=False)
_faixa_idade = _e["idade_dias"].map(lambda d: "sem data" if pd.isna(d) else "mesmo dia ou véspera" if d <= 1
                                    else "2 a 30 dias" if d <= 30 else "mais de um mês")
_ordem = ["mesmo dia ou véspera", "2 a 30 dias", "mais de um mês", "sem data"]
taxa_cruzada = (100 * _e.groupby([_faixa_cos, _faixa_idade], observed=True)["mesmo"].mean()).unstack().reindex(columns=_ordem).round(1)
contagem_cruzada = _e.groupby([_faixa_cos, _faixa_idade], observed=True).size().unstack().reindex(columns=_ordem)

_fig, _ax = plt.subplots(figsize=(8, 3.4))
_im = _ax.imshow(taxa_cruzada.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100, aspect="auto")
_ax.set_xticks(range(len(_ordem)), _ordem, fontsize=8)
_ax.set_yticks(range(len(taxa_cruzada.index)), taxa_cruzada.index, fontsize=8)
for _i in range(taxa_cruzada.shape[0]):                     # escreve o número em cada quadrado
    for _j in range(taxa_cruzada.shape[1]):
        _v, _n = taxa_cruzada.iat[_i, _j], contagem_cruzada.iat[_i, _j]
        _ax.text(_j, _i, "" if pd.isna(_v) else f"{_v:.0f}\n(n={int(_n)})", ha="center", va="center", fontsize=7,
                 color="white" if (not pd.isna(_v) and _v > 60) else "black")
_ax.set_title('De cada 100 fontes, quantas são o mesmo fato (estudo)')
_fig.colorbar(_im, ax=_ax, fraction=0.03)
_t = taxa_cruzada.loc["muito parecida (≥ 0,74)"]
_explicacao = (
    f"Só no estudo. Entre as fontes muito parecidas, são o mesmo fato {_t['mesmo dia ou véspera']:.0f} de cada 100 no mesmo "
    f"dia ou véspera, {_t['2 a 30 dias']:.0f} de 2 a 30 dias e {_t['mais de um mês']:.0f} acima de um mês: a mesma semelhança "
    "vale coisas diferentes conforme a idade. É o que a informação nova (cosseno × idade) conta ao modelo, que hoje só vê "
    "as duas separadas."
)
figura_cruzada = guard.fig(7, "semelhanca_por_idade", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_cruzada), width=640), mo.md(_explicacao), taxa_cruzada, contagem_cruzada])

# Ciclo 2, etapa 8: deixar tudo na mesma escala

**Em português:** o computador compara números, e eles precisam estar na mesma escala: idade em dias (até anos) e
semelhança (de -1 a 1) não se comparam sem uma conversão. A conversão é aprendida **só no estudo**; o simulado e a
prova só passam por ela. Neste ciclo, a conversão também calcula a informação nova, a semelhança vezes a idade.

| O que fazemos | Em português | Em termos técnicos |
|---|---|---|
| **Quatro versões** | com e sem a resposta final da regra; com e sem a informação nova (o simulado escolhe na etapa 10) | `montar_preprocessador(com_regra, interacao)` |
| **Provar que aprendeu só no estudo** | a mediana guardada para preencher a idade é a do estudo, não a do estudo somado ao simulado | checagem da mediana |
| **A figura** | estudo e simulado depois da conversão, sem o alvo | `08_*` |
| **A tabela pronta** | as fichas de todos os pares (estudo, simulado e prova) num arquivo com hash, para treinar no Mac | `entradas_do_decisor.csv` |

**O que eu espero, escrito antes de rodar:** o estudo fica com média 0 em cada número; o simulado, só rodada 3, fica
com o cosseno padronizado entre -0,20 e -0,05 (no ciclo 1, -0,10) e a idade entre +0,05 e +0,25 (+0,16); nada vazio;
a versão com a informação nova tem exatamente uma coluna a mais.

In [ ]:
import sys as _sys3
if str(mo.notebook_dir()) not in _sys3.path:
    _sys3.path.insert(0, str(mo.notebook_dir()))
from preprocessamento import montar_preprocessador

# As quatro versões. O nome começa com "preproc": é como a trava do pipeline reconhece pré-processamento.
preprocessadores = {(r, i): montar_preprocessador(com_regra=r, interacao=i) for r in (False, True) for i in (False, True)}
for preproc_versao in preprocessadores.values():
    preproc_versao.fit(estudo_entradas[preproc_versao.entradas])     # O AJUSTE: só com o estudo

preproc_base, preproc_nova = preprocessadores[(False, False)], preprocessadores[(False, True)]
X_estudo = preproc_nova.transform(estudo_entradas[preproc_nova.entradas])
X_simulado = preproc_nova.transform(simulado_entradas[preproc_nova.entradas])
nomes_das_colunas = list(preproc_nova.get_feature_names_out())

# CHECAGEM 1: a mediana guardada para a idade é a do ESTUDO (nas duas peças que usam a idade).
_log = lambda s: np.log1p(s.clip(lower=0))
_mediana_estudo = float(_log(estudo_entradas["idade_dias"]).median())
_mediana_com_simulado = float(_log(pd.concat([estudo_entradas["idade_dias"], simulado_entradas["idade_dias"]])).median())
_colunas = preproc_nova.named_steps["colunas"]
_m1 = float(_colunas.named_transformers_["idade"].named_steps["preenche"].statistics_[0])
_m2 = float(_colunas.named_transformers_["interacao"].named_steps["lado_a_lado"].named_transformers_["idade"]
            .named_steps["preenche"].statistics_[0])
assert abs(_m1 - _mediana_estudo) < 1e-9 and abs(_m2 - _mediana_estudo) < 1e-9, "a mediana não é a do estudo"
# CHECAGEM 2: nada vazio. CHECAGEM 3: a informação nova acrescenta exatamente uma coluna.
assert not np.isnan(X_estudo).any() and not np.isnan(X_simulado).any(), "sobrou vazio"
_base = len(preproc_base.get_feature_names_out())
assert len(nomes_das_colunas) == _base + 1 and nomes_das_colunas[-1] == "interacao__cosseno_x_idade"

resumo_preprocessamento = {
    "colunas: base / com a informação nova / com a regra / com as duas": [len(v.get_feature_names_out()) for v in preprocessadores.values()],
    "mediana da idade guardada (log)": round(_m1, 4),
    "mediana só do estudo (log)": round(_mediana_estudo, 4),
    "mediana se o simulado entrasse (log)": round(_mediana_com_simulado, 4),
}
resumo_preprocessamento

In [ ]:
# A FIGURA DA ETAPA 8: os números depois da conversão, estudo contra simulado. Sem o alvo.
_fig, _axes = plt.subplots(1, 3, figsize=(12, 3.2))
_medias = {}
for _ax, _nome, _titulo in [(_axes[0], "numeros__cosseno", "semelhança padronizada"),
                            (_axes[1], "idade__idade_dias", "idade padronizada"),
                            (_axes[2], "interacao__cosseno_x_idade", "semelhança × idade (nova)")]:
    _i = nomes_das_colunas.index(_nome)
    _ax.hist(X_estudo[:, _i], bins=40, alpha=0.6, density=True, label="estudo")
    _ax.hist(X_simulado[:, _i], bins=40, alpha=0.6, density=True, label="simulado")
    _ax.set_title(_titulo)
    _medias[_titulo] = (X_estudo[:, _i].mean(), X_simulado[:, _i].mean())
_axes[0].legend(fontsize=7)
_fig.tight_layout()
_c, _d = _medias["semelhança padronizada"], _medias["idade padronizada"]

def _comparado(diferenca, mais, menos, igual):   # a frase sai do número, nunca escrita à mão (a de 08/10 errou)
    return igual if abs(diferenca) < 0.05 else (mais if diferenca > 0 else menos)

_explicacao = (
    f"Depois da conversão aprendida só no estudo: o estudo fica com média {_c[0]:+.2f} na semelhança e {_d[0]:+.2f} na "
    f"idade; o simulado, só rodada 3, fica com {_c[1]:+.2f} e {_d[1]:+.2f}: "
    + _comparado(_c[1] - _c[0], "título e afirmação se parecem mais", "título e afirmação se parecem menos",
                 "título e afirmação se parecem tanto quanto no estudo")
    + " e "
    + _comparado(_d[1] - _d[0], "a fonte é mais velha", "a fonte é mais nova", "a fonte tem a idade do estudo")
    + ". A informação nova (à direita) é o produto das duas: alta quando a fonte é parecida e velha ao mesmo tempo."
)
figura_preproc = guard.fig(8, "preprocessamento_estudo_simulado", _fig, _explicacao, state_dir=mo.notebook_dir())
mo.vstack([mo.image(src=str(figura_preproc), width=760), mo.md(_explicacao)])

In [ ]:
import hashlib as _hashlib
import prepare as _p3

# A TABELA PRONTA: as fichas de todos os pares das rodadas 1 a 4, ordenadas pelo id, para os cadernos de treino
# rodarem em qualquer máquina (os cands-*.json só existem no PC). A prova entra porque a etapa 14 precisa das fichas
# dela; quem lê pelo dados.carregar() nunca a recebe.
_tabela = _p3.entradas_do_decisor((1, 2, 3, 4)).sort_index(kind="stable")
_arq = mo.notebook_dir() / "entradas_do_decisor.csv"
_arq.write_text(_tabela.to_csv(lineterminator="\n"), encoding="utf-8", newline="")   # o mesmo fim de linha em toda máquina
_hash = _hashlib.sha256(_arq.read_bytes()).hexdigest()
_meta = {"arquivo": "entradas_do_decisor.csv",
         "gerado_por": "prepare.entradas_do_decisor((1, 2, 3, 4)) no caderno 02_prep.py do ciclo 2 (precisa dos cands-*.json, só no PC)",
         "linhas": int(len(_tabela)), "colunas": list(_tabela.columns), "sha256": _hash}
(mo.notebook_dir() / "entradas_do_decisor.json").write_text(json.dumps(_meta, indent=1, ensure_ascii=False) + "\n",
                                                             encoding="utf-8", newline="\n")
tabela_pronta = {"linhas": _meta["linhas"], "colunas": len(_meta["colunas"]), "sha256": _hash[:16]}
tabela_pronta